<a href="https://colab.research.google.com/github/ikadekaggasugitha-cell/AI-InvestmentDecisionSupport/blob/main/RAKIMIN_HOMEWORK_DataScience.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## IMPORT LIBRARY AND READ FILE

In [ ]:
import pandas as pd

# Membaca data dari file CSV yang sudah ada
df = pd.read_csv('/content/online_retail_II.csv')

# Menampilkan 5 baris pertama data
display(df.head())

## **Section 1: Menganalisis Rata-Rata Pendapatan Per Tahun**

**1. Buat kolom baru dengan nama Year yang berisi nilai tahun dari Invoice Date**

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

#kolom baru 'Year' dengan mengambil nilai tahun dari InvoiceDate
df['Year'] = df['InvoiceDate'].dt.year

#beberapa baris data untuk verifikasi kolom baru
display(df[['InvoiceDate', 'Year']].head())

**2. Buat filtering data dengan ketentuan di bawah ini dan simpan dalam variabel baru**

In [ ]:
df['Invoice'] = df['Invoice'].astype(str)

#filtering data sesuai ketentuan
# - Quantity >= 1
# - Invoice tidak mengandung huruf 'C'
sales = df[(df['Quantity'] >= 1) & (~df['Invoice'].str.contains('C', na=False))]

#informasi jumlah baris setelah difilter dan beberapa sampel data
print(f"Jumlah data asli: {len(df)} baris")
print(f"Jumlah data setelah difilter (sales): {len(sales)} baris")
display(sales.head())

**3. Buat kolom baru bernama Revenue dengan nilai Quantity dikali dengan Price**

In [ ]:
# Membuat kolom baru 'Revenue' dengan mengalikan Quantity dan Price
sales['Revenue'] = sales['Quantity'] * sales['Price']

# Menampilkan data terbaru beserta kolom Revenue yang baru dibuat
display(sales[['Quantity', 'Price', 'Revenue']].head())

**4. Hitung rata-rata Revenue per tahun dan buat visualisasinya**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Hitung rata-rata Revenue per tahun
avg_revenue_per_year = sales.groupby('Year')['Revenue'].mean().reset_index()
print("Rata-rata Revenue per Tahun:")
display(avg_revenue_per_year)

# 2. Visualisasi dengan Bar Plot
plt.figure(figsize=(8, 5))
sns.barplot(x='Year', y='Revenue', data=avg_revenue_per_year, palette='Blues_d')

plt.title('Rata-rata Revenue Pelanggan per Tahun', fontsize=14)
plt.xlabel('Tahun (Year)', fontsize=12)
plt.ylabel('Rata-rata Revenue', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

**5. Tuliskan interpretasi-mu dari output nomor 4**

Berdasarkan hasil perhitungan rata-rata Revenue per tahun beserta visualisasi grafik batangnya, berikut adalah beberapa poin interpretasi :



1.   Kenaikan Konsisten Setiap Tahun:


*   Tahun 2009: Rata-rata transaksi menghasilkan nilai belanja sekitar 18.71 per transaksi.
*   Tahun 2010: Nilai ini meningkat menjadi sekitar 19.91.

*   Tahun 2011: Nilai ini kembali naik secara tipis menjadi sekitar 20.06.
*   Tren ini menunjukkan bahwa secara rata-rata, pelanggan cenderung membelanjakan uang lebih banyak di setiap transaksi baru seiring berjalannya tahun.




2.   Pertumbuhan Terbesar Terjadi Antara 2009 dan 2010:


*   Peningkatan rata-rata Revenue paling signifikan terlihat dari tahun 2009 ke tahun 2010 (naik sekitar 1.20 poin).
*   Sementara itu, pertumbuhan dari tahun 2010 ke 2011 cenderung lebih stabil dan mengalami saturasi (hanya naik sekitar 0.15 poin).



3.   Indikasi Kesehatan Bisnis yang Baik:



*   Karena kita sudah memfilter data dari pembatalan (Invoices dengan huruf 'C') dan kuantitas di bawah 1, data ini murni merepresentasikan transaksi sukses.
*   Kenaikan rata-rata nilai transaksi sukses mengindikasikan strategi penjualan yang baik, misalnya keberhasilan strategi cross-selling, up-selling, atau peningkatan harga jual yang diterima baik oleh pasar.







## **Section 2: Menganalisis Transaksi Pelanggan Per Tahun**

In [ ]:
# Melakukan filtering untuk menghapus Customer ID yang kosong (null/NaN)
finished = sales[sales['Customer ID'].notna()]

# Menampilkan jumlah baris sebelum dan sesudah filtering untuk verifikasi
print(f"Jumlah baris di data 'sales': {len(sales)}")
print(f"Jumlah baris di data 'finished' (tanpa Customer ID null): {len(finished)}")
display(finished.head())

In [ ]:
df['Invoice'] = df['Invoice'].astype(str)

#Filter transaksi yang dibatalkan (mengandung huruf 'C')
cancel = df[df['Invoice'].str.contains('C', na=False)]

#Tampilkan jumlah data yang dibatalkan dan beberapa baris sampel
print(f"Jumlah transaksi yang dibatalkan: {len(cancel)} baris")
display(cancel.head())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#hitung jumlah transaksi berhasil (finished) per tahun
# Menggunakan nunique('Invoice') untuk menghitung jumlah transaksi unik per tahun
finished_per_year = finished.groupby('Year')['Invoice'].nunique().reset_index()
finished_per_year.rename(columns={'Invoice': 'Completed_Transactions'}, inplace=True)

#Menghitung jumlah transaksi batal (cancel) per tahun
cancel_per_year = cancel.groupby('Year')['Invoice'].nunique().reset_index()
cancel_per_year.rename(columns={'Invoice': 'Canceled_Transactions'}, inplace=True)

# Merge kedua data untuk perbandingan
transaction_summary = pd.merge(finished_per_year, cancel_per_year, on='Year', how='outer').fillna(0)
print("Ringkasan Jumlah Transaksi Per Tahun:")
display(transaction_summary)

#Restrukturisasi data untuk mempermudah plotting dengan Seaborn
data_melted = pd.melt(transaction_summary, id_vars=['Year'],
                      value_vars=['Completed_Transactions', 'Canceled_Transactions'],
                      var_name='Status', value_name='Total_Transactions')

#Membuat Visualisasi Grafik
plt.figure(figsize=(10, 6))
sns.barplot(x='Year', y='Total_Transactions', hue='Status', data=data_melted, palette='Set2')

plt.title('Perbandingan Jumlah Transaksi Berhasil vs Dibatalkan Per Tahun', fontsize=14)
plt.xlabel('Tahun (Year)', fontsize=12)
plt.ylabel('Jumlah Transaksi Unik (Invoice)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Status Transaksi')

plt.tight_layout()
plt.show()

In [ ]:
#hitung jumlah customer unik yang melakukan pembatalan (dari data cancel)
cancel_customers = cancel[cancel['Customer ID'].notna()].groupby('Year')['Customer ID'].nunique().reset_index()
cancel_customers.rename(columns={'Customer ID': 'Unique_Canceled_Customers'}, inplace=True)

#Menghitung total seluruh customer unik (gabungan finished dan cancel)
#menggabungkan data 'finished' dan 'cancel' untuk mendapatkan semua customer unik per tahun
all_transactions = pd.concat([finished, cancel[cancel['Customer ID'].notna()]])
total_customers = all_transactions.groupby('Year')['Customer ID'].nunique().reset_index()
total_customers.rename(columns={'Customer ID': 'Total_Unique_Customers'}, inplace=True)

#Menggabungkan data dan menghitung Cancellation Rate
cancellation_rate_df = pd.merge(total_customers, cancel_customers, on='Year', how='left').fillna(0)
cancellation_rate_df['Cancellation_Rate (%)'] = (
    cancellation_rate_df['Unique_Canceled_Customers'] / cancellation_rate_df['Total_Unique_Customers']
) * 100

# Menampilkan hasil perhitungan
print("Hasil Perhitungan Cancellation Rate Per Tahun:")
display(cancellation_rate_df)

### **Interpretasi Perbandingan Analisis Transaksi dan Cancellation Rate**

Berdasarkan hasil analisis dari **Output Nomor 3 (Jumlah Transaksi)** dan **Output Nomor 4 (Cancellation Rate)**, didapatkan beberapa temuan penting sebagai berikut:

#### **1. Korelasi Skala Bisnis dan Tingkat Pembatalan (Tahun 2009 vs 2010)**
* **Fakta:** Pada tahun 2010, bisnis mengalami lonjakan volume transaksi unik yang sangat besar (dari 1.512 transaksi sukses menjadi 18.327). Namun, lonjakan ini juga diiringi dengan peningkatan tajam **Cancellation Rate** dari **29,57%** menjadi **40,34%**.
* **Interpretasi:** Skala bisnis yang tumbuh sangat cepat dalam waktu singkat tampaknya tidak diimbangi dengan kesiapan operasional yang matang. Angka pembatalan yang mencapai 40,34% (hampir setengah dari total pelanggan unik melakukan pembatalan minimal sekali) mengindikasikan adanya masalah potensial seperti ketidaktersediaan stok barang, sistem pembayaran yang bermasalah, atau keterlambatan proses pengiriman.

#### **2. Indikasi Perbaikan Kualitas Layanan (Tahun 2010 vs 2011)**
* **Fakta:** Pada tahun 2011, meskipun total transaksi sukses sedikit menurun (dari 18.327 menjadi 17.136), jumlah transaksi batal turun lebih signifikan secara proporsional (dari 4.381 menjadi 3.510). Hal ini berhasil menekan **Cancellation Rate** turun menjadi **35,20%**.
* **Interpretasi:** Penurunan tingkat pembatalan ini menunjukkan adanya peningkatan kualitas layanan pelanggan atau efisiensi sistem operasional. Meskipun jumlah basis pelanggan aktif (`Total_Unique_Customers`) relatif stabil di angka sekitar 4.200-an, lebih sedikit pelanggan yang melakukan pembatalan pesanan di tahun 2011 dibandingkan tahun sebelumnya. Ini merupakan sinyal positif bagi retensi pelanggan.

#### **3. Kesimpulan Bisnis**
* Secara keseluruhan, meskipun bisnis menunjukkan pertumbuhan volume transaksi yang sangat baik sejak 2009, angka pembatalan yang berada di atas 30% tetap memerlukan perhatian khusus. Manajemen disarankan untuk melakukan evaluasi lebih dalam mengenai alasan pembatalan utama guna mengoptimalkan potensi pendapatan yang hilang (*lost revenue*).